# Industry Factors — Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for building the
industry factor exposures as MSCI describes them in *The Barra US Equity Model
(USE4)*: 60 GICS-based industry factors with 0/1 (and, for diversified firms,
fractional) exposures. Public data forces material engineering — documented
exhaustively in §9 — yielding **55 factors** over Sharadar's classification.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output as deeply untrustworthy until audited. Follow the stages linearly. Each stage has:
- ✅ **PDF SPEC** — what USE4 specifies
- ❓ **NOT IN PDF** / **NOT IN DATA** — judgment calls and data gaps
- 💡 **DEFAULT** — the chosen resolution
- 🧪 **VALIDATE** — sanity checks

**A note on replicating 2011 in 2026.** USE4's industry set was engineered for the
2011 market. Sector tectonics have moved since (software and internet names are an
order of magnitude larger; coal, papers, department stores have hollowed out), so a
modern rebuild *should not* reproduce the 60-list verbatim even where it could —
the published selection criteria (economic intuition, statistical significance,
explanatory power, **minimum size**) matter more than the labels. Our merges and
splits below are those criteria applied to the 2026 universe.

**Inputs:** `SHARADAR_TICKERS.csv` (Sharadar ticker master — classification),
`data/out/estu_monthly.parquet`, `industry_scheme.csv` (this directory — the
hand-engineered mapping, version-controlled and reviewable).

**Deliverables:** `data/out/industries_use4.parquet` (one industry-factor
label per stock per date) and `data/out/industry_weights_use4.parquet`
(the per-date industry cap-weight vector for the CSR's cap-weighted zero-sum
constraint on industry returns).

## §1. The USE4 industry spec — what the documentation says

> The USE4 model contains **60 industry factors built on GICS**. The structure is a
> hybrid of GICS levels: some factors correspond to a single GICS industry or
> sub-industry, others aggregate or split GICS categories based on **economic
> intuition, statistical significance, explanatory power, and minimum size**.
> Thin industries are avoided because they produce unstable estimates.

> **Exposures:** most stocks have one primary industry exposure of 1.0. For
> diversified firms, USE4 estimates fractional exposures from business-segment
> reporting: industry-specific price-to-assets and price-to-sales coefficients
> from cross-sectional regressions, blended **75% assets / 25% sales**, top five
> segments retained and renormalized. ~63% of the estimation universe by cap
> weight had multiple exposures.

> **Identification:** every stock has country exposure 1 and industry exposures
> summing to 1, so the cross-sectional regression imposes a **cap-weighted
> zero-sum constraint on industry factor returns** — industry returns are pure
> performance relative to the market.

---

**What this spec builds is the exposure layer only.** The constraint and the
factor returns belong to the cross-sectional-regression stage (next repo);
the deliverable here must merely be regression-ready (complete coverage,
exposures summing to 1 — trivially true under single membership).

## §2. End-to-end pipeline

```
┌────────────────────────────────────────────────────────────────────┐
│  UPSTREAM:  estu_build.ipynb  →  estu_monthly.parquet              │
│             industry_scheme.csv (hand-engineered, this directory)  │
├────────────────────────────────────────────────────────────────────┤
│  STAGE 0:  Setup, parameters, floor rule                           │
│  STAGE 1:  Load scheme + TICKERS classification                    │
│  STAGE 2:  Load shared ESTU                                        │
│  STAGE 3:  Classify: permaticker → Sharadar industry → factor      │
│  STAGE 4:  Panel construction (static map × monthly calendar)      │
│  STAGE 5:  Save deliverable                                        │
│  STAGE 6:  Validate (8-check battery)                              │
└────────────────────────────────────────────────────────────────────┘
```

Industries are **dummies, not descriptors**: no trim, no standardisation, no
MoM ρ. The validation battery is membership-shaped (coverage, floor, churn,
concentration) instead.

**One-sweep run:** `your end-to-end runner` (stage `industries`,
after `estu`; independent of `daily` and the styles).

## §3. Output schema — what the worker delivers

**File:** `data/out/industries_use4.parquet`

**Compression:** zstd, statistics=True

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `signal_date` | Date | End-of-month rebalance date |
| `in_estu` | Bool | ESTU membership on signal_date |
| `mcap` | Float64 | Market cap on signal_date |
| `sharadar_industry` | String | The raw classification atom (audit column) |
| `industry` | String | **The factor label** — one of the 55 scheme factors |
| `use4_sector` | String | Sector grouping of the factor |

**The scheme itself** lives in `industry_scheme.csv`: 151 Sharadar `industry`
atoms → 55 factors (+1 `UNASSIGNED` fallback), with sector and floor-exception
flags. The CSV is the source of truth — the build loads it, never hardcodes it.

**Coverage rules:**
- Every row of `estu_monthly` gets a label (coverage universe, not just ESTU).
- Single membership: exactly one row per (permaticker, signal_date); the one-hot
  exposure matrix for the CSR is a pivot away.
- Exposures sum to 1 per stock by construction.

### Second artifact: `industry_weights_use4.parquet`

| Column | Type | Description |
|---|---|---|
| `signal_date` | Date | End-of-month rebalance date |
| `industry` | String | Factor label |
| `n_members` | UInt32 | In-ESTU members on the date |
| `cap_weight` | Float64 | Industry share of total in-ESTU cap; sums to 1 per date (asserted < 1e-12) |

This is the $w$-vector for USE4's identification constraint
$\sum_j w_j \, f_j^{\text{ind}} = 0$ — materialised here so the CSR (and the
country-factor spec) never re-derive it ad hoc.


## §4. STAGE 0 — Setup, parameters, floor rule

```python
# ───── Parameters ─────────────────────────────────────────────────────────────
SCHEME_PATH    = Path(__file__ or ".") / "industry_scheme.csv"   # adjacent to notebook
CALENDAR_START = date(1999, 1, 1)

# 💡 DEFAULT (NOT IN PDF) — minimum-size floor for a viable factor
FLOOR_MEDIAN = 15   # median ESTU members over the full sample
FLOOR_MIN    = 8    # never fewer at any single date
# Declared exceptions (kept despite the floor; justification in §9):
FLOOR_EXCEPTIONS = {"Internet & Catalog Retail", "Managed Health Care",
                    "Airlines", "Industrial Conglomerates"}
```

❓ **NOT IN PDF — the floor values.** USE4 says only "minimum size".
💡 **DEFAULT:** median ≥ 15 and min ≥ 8 ESTU members over 1999–2026. With our
~2,500-name ESTU and 55 factors this averages ~45 members per factor.

## §5. STAGE 1 — Scheme + classification load

✅ **PDF SPEC:** GICS is the classification foundation.

❌ **NOT IN DATA:** GICS is licensed and absent from Sharadar.

💡 **DEFAULT:** Sharadar's `industry` field (TICKERS; 151 observed values,
Morningstar-style taxonomy, 100% ESTU coverage) is the atomic layer; the
hand-engineered `industry_scheme.csv` aggregates the atoms into 55 factors
targeting the published USE4 60-list. `famaindustry` and `sicsector` were
evaluated and rejected (too coarse: FF48's "Business Services" holds 328
members; SIC's tech granularity is 1980s-vintage).

🧪 **VALIDATE:**
- Scheme covers exactly the observed atom set — unmapped atom ⇒ **fail loudly**
  (a vendor taxonomy change must be reviewed, never silently bucketed)
- 55 factors + 1 fallback; every factor has ≥1 atom
- TICKERS dedup: one row per permaticker

## §6. STAGE 2 — Load the shared ESTU (`estu_monthly.parquet`)

✅ **PDF SPEC:** exposures are estimated for the estimation universe.

💡 **DEFAULT:** the shared `data/out/estu_monthly.parquet` from
`estu_build.ipynb` — same universe as every style factor, which the joint
cross-sectional regression requires.

🧪 **VALIDATE:** 330 monthly dates (1999-01-29 →), ESTU size mean ≈ 2,500.

## §7. STAGES 3–4 — Classification and panel

✅ **PDF SPEC:** one primary industry exposure of 1.0 per stock; fractional for
diversified firms via segment data (75% assets / 25% sales).

❌ **NOT IN DATA:** SF1 carries no business-segment reporting — the fractional
machinery is unbuildable. **Single 0/1 membership for every stock.** This is a
headline deviation (USE4 reports ~63% of cap weight multi-exposed); the bias
concentrates in conglomerates, diversified financials, and integrated energy.

❓ **NOT IN PDF — classification timing.** TICKERS is a current snapshot, not
point-in-time: today's label applied to 1999 introduces mild look-ahead for
reclassified/pivoted companies.
💡 **DEFAULT:** static classification, documented; **revisit trigger**: a PIT
layer from EDGAR 10-K header SIC codes (free, filing-dated) if classification
drift is shown to matter.

**Mechanics:** map each ESTU-panel row permaticker → `industry` atom → factor.
Null atom → `UNASSIGNED` (historically ≤1 member at any date; asserted ≤2).
The map is static, so the panel is the map crossed with the monthly calendar.

🧪 **VALIDATE:**
- 100% of rows labelled; in-ESTU `UNASSIGNED` ≤ 2 per date
- Exactly one row per (permaticker, signal_date)
- Per-permaticker label is constant over time (static map ⇒ churn only via
  universe entry/exit; any other churn is a bug)

## §8. STAGES 5–6 — Save and validate

**Save:** zstd parquet, 7-column dtype assertion, read-back assertion.

**The 8-check battery:**

```
1   Coverage          in-ESTU UNASSIGNED rows ≤ 2 at every date
2   Completeness      all 55 factors have ≥ 1 ESTU member at every date
3   Floor             non-exception factors: median ≥ 15 and min ≥ 8 members;
                      the 4 exceptions reported with their stats
4   Static map        every permaticker carries exactly one factor label
5   Single membership exactly one row per (permaticker, signal_date)
6   Concentration     largest factor ≤ 30% of ESTU cap at every date
7   Spot checks       AAPL→Computers & Electronics; JPM→Banks;
                      XOM→Oil, Gas & Consumable Fuels; AMZN→Internet & Catalog
                      Retail; NVDA→Semiconductors; PLD→Real Estate
8   Disk ≡ memory     read-back equivalence
```

**Shared validation conventions** apply where meaningful (the battery runs on
the saved deliverable; there is no in-progress-month coverage issue because the
map is static).

## §9. Master list of ❓ NOT-IN-PDF / ❌ NOT-IN-DATA decisions

| # | Decision | Default chosen | Revisit when |
|---|---|---|---|
| 1 | GICS unavailable | Sharadar `industry` atoms (151) + hand-engineered 55-factor scheme targeting the USE4 60-list | GICS or a crosswalk is licensed |
| 2 | No segment data | Single 0/1 membership for all stocks (USE4: ~63% of cap multi-exposed) | Segment-level data procured |
| 3 | Classification not PIT | Static current-snapshot labels | EDGAR 10-K SIC PIT layer if drift shown to matter |
| 4 | Floor rule | median ≥ 15 and min ≥ 8 over full sample | CSR-based explanatory-power screening replaces member counts |
| 5 | Floor exceptions (4) | Internet & Catalog Retail (AMZN, $3.0T), Managed Health Care (USE4-explicit), Airlines, Industrial Conglomerates | Any falls below ~5 members persistently |
| 6 | Merges vs the 60-list | Drilling→Equip&Svcs; Paper+Forest→Construction Materials; both metals→Metals & Mining; durables/apparel/leisure consolidated; retail consolidated; telecom 2→1 (no wireless split in atoms) | CSR explanatory power favors a split AND members support it |
| 7 | Splits vs the 60-list | Diversified Financials → Capital Markets & AM / Consumer Finance & DFS; Software → Application / Infrastructure & Gaming (2026 tectonics: these were 107–151-member monsters) | Never on fidelity grounds — these are deliberate modernisations |
| 8 | Fat factors kept whole | Banks (med 194), Real Estate (155), Biotech & Life Sciences (128), Software-App (115) per USE4-2011 | CSR screening adjudicates further splits |
| 9 | Judgment atoms | Solar→Semiconductor Equipment; Consumer Electronics (incl. AAPL)→Computers & Electronics; Shell Companies (SPACs)→Consumer Finance & DFS; Education & Training→Hotels & Consumer Services | Reclassification evidence |
| 10 | REITs in scope | One Real Estate factor under Financials (USE4-2011, pre-2016 GICS promotion) | Moving to a modern GICS baseline |

## §10. Common pitfalls — read this before coding

**Pitfall 1: silently bucketing unmapped atoms.** A Sharadar taxonomy update can
introduce new `industry` values. The build must FAIL on an unmapped atom, not
default it — the scheme CSV is reviewed, code defaults are not.

**Pitfall 2: classification churn masquerading as universe churn.** With a static
map, a permaticker's factor can never change. If membership counts jump without
ESTU entry/exit, the join is broken (duplicate TICKERS rows are the usual cause —
dedup per permaticker first).

**Pitfall 3: treating the fallback as a factor.** `UNASSIGNED` rows are kept in
the deliverable (audit trail) but must be excluded from the CSR exposure matrix.

**Pitfall 4: forgetting the constraint downstream.** Country exposure ≡ 1 plus
single industry membership ⇒ perfect collinearity. The CSR must impose the
cap-weighted zero-sum constraint on industry returns; nothing in THIS deliverable
prevents the mistake.

**Pitfall 5: judging factors by member count alone.** Internet & Catalog Retail
has ~6 median members and $3T of cap. Cap weight is the relevant size for a
cap-weighted regression; the member floor guards estimate stability, not
importance.

## §11. Final summary — what "done" looks like

1. ✅ `industries_use4.parquet` matches the §3 schema
2. ✅ All 8 validation checks pass
3. ✅ Exactly 55 factors, every atom of `industry_scheme.csv` consumed
4. ✅ The four floor exceptions reported with their member stats
5. ✅ Spot checks land (AAPL, JPM, XOM, AMZN, NVDA, PLD)

Downstream: the CSR repo pivots `industry` to one-hot, joins the style
exposures, adds the country column of ones, and imposes the cap-weighted
zero-sum industry constraint.